# ️ Glu-Stock: 04_PERFORMANCE_LAB
**Phase**: Historical Performance Analysis | v18.26 (Dual Core Sync)

This notebook simulates signals based on the **CNN-Only Technical Gate** to validate the impact of decommissioning LGBM on signal recall and precision.

In [ ]:
#!pip install -q yfinance pandas tensorflow ta


In [ ]:
import os, json, numpy as np, pandas as pd, yfinance as yf, warnings, ta
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()
    def predict(self, df):
        try:
            data = df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(30).values
            norm = (data - data.min(axis=0)) / (data.max(axis=0) - data.min(axis=0) + 1e-7)
            self.interpreter.set_tensor(self.interpreter.get_input_details()[0]['index'], np.expand_dims(norm.astype(np.float32), axis=0))
            self.interpreter.invoke()
            out = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(out[1]) if len(out) > 1 else float(out[0])
        except: return 0.5


In [ ]:
def simulate_trade_exit(future_df, entry_price, tp=0.03, sl=0.02, horizon=10):
    if len(future_df) == 0: return 0.0, 0
    tpp, slp = entry_price * (1 + tp), entry_price * (1 - sl)
    for i in range(min(len(future_df), horizon)):
        if future_df['High'].iloc[i] >= tpp: return tp * 100, i + 1
        if future_df['Low'].iloc[i] <= slp: return -sl * 100, i + 1
    final_ret = (future_df['Close'].iloc[min(len(future_df)-1, horizon-1)] / entry_price - 1) * 100
    return final_ret, min(len(future_df), horizon)

def run_dual_core_backtest():
    print("Starting 2025 CNN-Only Backtest (v18.26)...\n")
    cnn_path = '/kaggle/input/notebooks/permanalwep/glu-stock-cnn-00b/cnn_daily_t2.tflite'
    if not os.path.exists(cnn_path): 
        print(f"[ERR] Model not found: {cnn_path}")
        return
    cnn = CNNPredictor(cnn_path)
    
    cohort = ['BBCA.JK', 'TLKM.JK', 'ASII.JK', 'ADRO.JK', 'BMRI.JK', 'BBRI.JK', 'ICBP.JK', 'PTBA.JK', 'ANTM.JK', 'UNTR.JK']
    raw_data = yf.download(cohort + ['^JKSE'], start='2024-06-01', end='2025-12-31', progress=False, auto_adjust=True)
    
    valid_dates = [d for d in raw_data.index if d.year == 2025]
    logs = []
    
    print(f"Scanning {len(valid_dates)} trading days...")
    for date in valid_dates:
        for ticker in cohort:
            try:
                hist = raw_data.loc[:date, (slice(None), ticker)].dropna()
                hist.columns = hist.columns.droplevel(1)
                if len(hist) < 150: continue
                if hist['Close'].iloc[-1] > hist['Close'].tail(50).mean():
                    score = cnn.predict(hist)
                    if score >= 0.45:
                        entry = float(hist['Close'].iloc[-1])
                        future = raw_data.loc[date + timedelta(days=1):, (slice(None), ticker)]
                        future.columns = future.columns.droplevel(1)
                        pnl, days = simulate_trade_exit(future, entry)
                        logs.append({'Date': date.strftime('%Y-%m-%d'), 'Ticker': ticker, 'Score': f"{score:.2%}", 'PnL%': pnl, 'HoldDays': days})
            except: continue
            
    df = pd.DataFrame(logs)
    if df.empty: print("\n[FAIL] No signals found."); return
    
    wr = (df['PnL%'] > 0).mean() * 100
    print(f"\n--- CNN-ONLY PERFORMANCE (2025) ---")
    print(f"Total Signals: {len(df)}")
    print(f"Win Rate     : {wr:.1f}%")
    print(f"Avg Profit   : {df['PnL%'].mean():.2f}%")
    print("\n--- Detailed Results ---")
    print(df.head(50).to_string(index=False))

run_dual_core_backtest()
